In [ ]:
import random
import shutil
import os
import json
from collections import defaultdict
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm

분류할 클래스의 JSON 이외는 삭제

In [ ]:
json_root_path = r'C:\Users\thsdu\Desktop\초급 프로젝트\sprint_ai_project1_data\train_annotations'

# 매핑 정보 (텍스트를 바로 사전으로 변환)
mapping_text = """
1900: 0: 보령부스파정 5mg
2483: 1: 뮤테란캡슐 100mg
3351: 2: 일양하이트린정 2mg
3483: 3: 기넥신에프정(은행엽엑스)(수출용)
3544: 4: 무코스타정(레바미피드)(비매품)
3743: 5: 알드린정
3832: 6: 뉴로메드정(옥시라세탐)
4543: 7: 에어탈정(아세클로페낙)
12081: 8: 리렉스펜정 300mg/PTP
12247: 9: 아빌리파이정 10mg
12778: 10: 다보타민큐정 10mg/병
13395: 11: 써스펜8시간이알서방정 650mg
13900: 12: 에빅사정(메만틴염산염)(비매품)
16232: 13: 리피토정 20mg
16262: 14: 크레스토정 20mg
16548: 15: 가바토파정 100mg
16551: 16: 동아가바펜틴정 800mg
16688: 17: 오마코연질캡슐(오메가-3-산에틸에스테르90)
18147: 18: 리리카캡슐 150mg
18357: 19: 종근당글리아티린연질캡슐(콜린알포세레이트)
19232: 20: 콜리네이트연질캡슐 400mg
19552: 21: 트루비타정 60mg/병
19607: 22: 스토가정 10mg
19861: 23: 노바스크정 5mg
20014: 24: 마도파정
20238: 25: 플라빅스정 75mg
20877: 26: 엑스포지정 5/160mg
21325: 27: 아토르바정 10mg
21771: 28: 라비에트정 20mg
22074: 29: 리피로우정 20mg
22347: 30: 자누비아정 50mg
22362: 31: 맥시부펜이알정 300mg
24850: 32: 놀텍정 10mg
25367: 33: 자누메트정 50/850mg
25438: 34: 큐시드정 31.5mg/PTP
25469: 35: 아모잘탄정 5/100mg
27733: 36: 트윈스타정 40/5mg
27777: 37: 카나브정 60mg
27926: 38: 울트라셋이알서방정
27993: 39: 졸로푸트정 100mg
28763: 40: 트라젠타정(리나글립틴)
29345: 41: 비모보정 500/20mg
29451: 42: 레일라정
29667: 43: 리바로정 4mg
30308: 44: 트라젠타듀오정 2.5/850mg
31863: 45: 아질렉트정(라사길린메실산염)
31885: 46: 자누메트엑스알서방정 100/1000mg
32310: 47: 글리아타민연질캡슐
33009: 48: 신바로정
33208: 49: 에스원엠프정 20mg
33880: 50: 글리틴정(콜린알포세레이트)
34597: 51: 제미메트서방정 50/1000mg
35206: 52: 아토젯정 10/40mg
36637: 53: 로수젯정10/5밀리그램
38162: 54: 로수바미브정 10/20mg
41768: 55: 카발린캡슐 25mg
"""

# 유효한 ID를 세트로
valid_ids = set()
for line in mapping_text.strip().split('\n'):
    if line:
        pill_id = int(line.split(':')[0].strip())
        valid_ids.add(pill_id)

deleted_count = 0
kept_count = 0

# 이미지 조합 폴더들 탐색
for combo_folder in os.listdir(json_root_path):
    combo_path = os.path.join(json_root_path, combo_folder)
    
    if os.path.isdir(combo_path) and combo_folder.endswith('_json'):
        for class_folder in os.listdir(combo_path):
            class_path = os.path.join(combo_path, class_folder)
            
            if os.path.isdir(class_path) and class_folder.startswith('K-'):
                try:
                    class_id = int(class_folder.split('-')[1])
                    
                    if class_id in valid_ids:
                        kept_count += 1
                    else:
                        shutil.rmtree(class_path)
                        deleted_count += 1
                except (IndexError, ValueError):
                    print(f"폴더명 분석 불가: {class_folder}")

print(f" - 유지된 클래스 폴더: {kept_count}개")
print(f" - 삭제된 클래스 폴더: {deleted_count}개")

클래스ID 약 종류에 따라 맞추기

In [ ]:
json_root_path = r'C:\Users\thsdu\Desktop\초급 프로젝트\sprint_ai_project1_data\train_annotations'

# 매핑 정보
mapping_text = """
1900: 0: 보령부스파정 5mg
2483: 1: 뮤테란캡슐 100mg
3351: 2: 일양하이트린정 2mg
3483: 3: 기넥신에프정(은행엽엑스)(수출용)
3544: 4: 무코스타정(레바미피드)(비매품)
3743: 5: 알드린정
3832: 6: 뉴로메드정(옥시라세탐)
4543: 7: 에어탈정(아세클로페낙)
12081: 8: 리렉스펜정 300mg/PTP
12247: 9: 아빌리파이정 10mg
12778: 10: 다보타민큐정 10mg/병
13395: 11: 써스펜8시간이알서방정 650mg
13900: 12: 에빅사정(메만틴염산염)(비매품)
16232: 13: 리피토정 20mg
16262: 14: 크레스토정 20mg
16548: 15: 가바토파정 100mg
16551: 16: 동아가바펜틴정 800mg
16688: 17: 오마코연질캡슐(오메가-3-산에틸에스테르90)
18147: 18: 리리카캡슐 150mg
18357: 19: 종근당글리아티린연질캡슐(콜린알포세레이트)
19232: 20: 콜리네이트연질캡슐 400mg
19552: 21: 트루비타정 60mg/병
19607: 22: 스토가정 10mg
19861: 23: 노바스크정 5mg
20014: 24: 마도파정
20238: 25: 플라빅스정 75mg
20877: 26: 엑스포지정 5/160mg
21325: 27: 아토르바정 10mg
21771: 28: 라비에트정 20mg
22074: 29: 리피로우정 20mg
22347: 30: 자누비아정 50mg
22362: 31: 맥시부펜이알정 300mg
24850: 32: 놀텍정 10mg
25367: 33: 자누메트정 50/850mg
25438: 34: 큐시드정 31.5mg/PTP
25469: 35: 아모잘탄정 5/100mg
27733: 36: 트윈스타정 40/5mg
27777: 37: 카나브정 60mg
27926: 38: 울트라셋이알서방정
27993: 39: 졸로푸트정 100mg
28763: 40: 트라젠타정(리나글립틴)
29345: 41: 비모보정 500/20mg
29451: 42: 레일라정
29667: 43: 리바로정 4mg
30308: 44: 트라젠타듀오정 2.5/850mg
31863: 45: 아질렉트정(라사길린메실산염)
31885: 46: 자누메트엑스알서방정 100/1000mg
32310: 47: 글리아타민연질캡슐
33009: 48: 신바로정
33208: 49: 에스원엠프정 20mg
33880: 50: 글리틴정(콜린알포세레이트)
34597: 51: 제미메트서방정 50/1000mg
35206: 52: 아토젯정 10/40mg
36637: 53: 로수젯정10/5밀리그램
38162: 54: 로수바미브정 10/20mg
41768: 55: 카발린캡슐 25mg
"""

pill_to_yolo = {}
for line in mapping_text.strip().split('\n'):
    parts = line.split(':')
    p_id = int(parts[0].strip())      
    y_id = int(parts[1].strip())     
    pill_to_yolo[p_id] = y_id

# JSON 순회 및 수정
updated_files = 0

for combo_dir in os.listdir(json_root_path):
    combo_path = os.path.join(json_root_path, combo_dir)
    if not os.path.isdir(combo_path): continue
    
    for class_dir in os.listdir(combo_path):
        class_path = os.path.join(combo_path, class_dir)
        if not os.path.isdir(class_path): continue

        try:
            pill_id = int(class_dir.split('-')[1])
            new_id = pill_to_yolo.get(pill_id)
            
            if new_id is not None:
                for json_file in os.listdir(class_path):
                    if json_file.endswith('.json'):
                        file_path = os.path.join(class_path, json_file)
                        
                        with open(file_path, 'r', encoding='utf-8') as f:
                            data = json.load(f)
                        for ann in data.get('annotations', []):
                            ann['category_id'] = new_id
                        with open(file_path, 'w', encoding='utf-8') as f:
                            json.dump(data, f, indent=4, ensure_ascii=False)
                        
                        updated_files += 1
        except (IndexError, ValueError):
            continue

print(f" - 총 {updated_files}개의 JSON 파일의 category_id가 수정됨")

JSON 파일 내 annotation 정보 확인

In [ ]:
# 경로 설정
json_root_path = r'C:\Users\thsdu\Desktop\초급 프로젝트\sprint_ai_project1_data\train_annotations'

deleted_files = 0
checked_files = 0

 # 디렉토리 순회
for combo_dir in os.listdir(json_root_path):
    combo_path = os.path.join(json_root_path, combo_dir)
    if not os.path.isdir(combo_path): continue
    
    for class_dir in os.listdir(combo_path):
        class_path = os.path.join(combo_path, class_dir)
        if not os.path.isdir(class_path): continue
        
        for json_file in os.listdir(class_path):
            if json_file.endswith('.json'):
                file_path = os.path.join(class_path, json_file)
                checked_files += 1
                
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                     
                    # annotations 리스트가 아예 없거나 비어있는 경우
                    # annotations가 있어도 첫 번째 항목에 bbox 정보가 없는 경우
                    annotations = data.get('annotations', [])
                    
                    is_empty = False
                    if not annotations:
                        is_empty = True
                    else:
                        # bbox 리스트가 비어있거나, 모든 좌표가 0인 경우 등 체크
                        first_bbox = annotations[0].get('bbox', [])
                        if not first_bbox or sum(first_bbox) == 0:
                            is_empty = True
                    
                    if is_empty:
                        os.remove(file_path)
                        deleted_files += 1
                        
                except (json.JSONDecodeError, PermissionError) as e:
                    print(f"파일 처리 오류 ({json_file}): {e}")

print(f" - 총 검사 파일: {checked_files}개")
print(f" - 삭제된 빈 데이터 파일: {deleted_files}개")

각 JSON 파일을 하나의 이미지당 하나의 JSON 파일이 나올 수 있도록 병합

In [ ]:
from collections import defaultdict

# 경로 설정
json_root_path = r'C:\Users\thsdu\Desktop\초급 프로젝트\sprint_ai_project1_data\train_annotations'
output_merged_path = r'C:\Users\thsdu\Desktop\초급 프로젝트\sprint_ai_project1_data\train_merge_annotations'
os.makedirs(output_merged_path, exist_ok=True)

# 이미지 파일명을 키(Key)로 하여 데이터 그룹화
image_groups = defaultdict(list)
image_metadata = {}


for root, dirs, files in os.walk(json_root_path):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    data = json.load(f)
                    
                    # JSON 내부의 실제 이미지 파일명 추출
                    if 'images' in data and len(data['images']) > 0:
                        img_name = data['images'][0]['file_name']
                        # 확장자를 .json으로 바꿔서 저장될 파일명 결정
                        json_name = os.path.splitext(img_name)[0] + ".json"
                        
                        # 해당 이미지의 metadata 최초 1회 저장
                        if json_name not in image_metadata:
                            image_metadata[json_name] = data['images'][0]
                        
                        # annotations만 따로 모으기
                        if 'annotations' in data:
                            image_groups[json_name].extend(data['annotations'])
                except Exception as e:
                    print(f"파일 읽기 오류 ({file}): {e}")

print(f"총 {len(image_groups)}개의 이미지 데이터 그룹")

# 그룹화된 데이터를 하나의 JSON 파일로 병합 및 저장
merged_count = 0

for json_name, all_anns in image_groups.items():
    # 최종 JSON 구조 생성
    final_json = {
        "images": [image_metadata[json_name]],
        "type": "instances",
        "annotations": all_anns,
        "categories": [
            {"supercategory": "pill", "id": 1, "name": "Drug"} # 기본값 유지
        ]
    }
    
    # 통합된 annotations 리스트 내에서 개별 객체 ID 재정렬
    for idx, ann in enumerate(final_json['annotations']):
        ann['id'] = idx + 1
        ann['image_id'] = image_metadata[json_name]['id']

    save_path = os.path.join(output_merged_path, json_name)
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(final_json, f, indent=4, ensure_ascii=False)
    
    merged_count += 1

print(f" - 생성된 통합 JSON 개수: {merged_count}개")
print(f" - 저장 위치: {output_merged_path}")

annonations 확인하여 class가 중복해서 들어가는 상황확인

In [ ]:
from collections import Counter

json_root = r'C:\Users\thsdu\Desktop\초급 프로젝트\sprint_ai_project1_data\train_merge_annotations'

deleted_count = 0
total_checked = 0

# os.walk를 사용하여 모든 하위 폴더 내의 JSON 파일을 탐색
for root, dirs, files in os.walk(json_root):
    for file in files:
        if file.lower().endswith('.json'):
            total_checked += 1
            json_path = os.path.join(root, file)
            
            try:
                # 파일 읽기
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                

                category_ids = [ann.get('category_id') for ann in data.get('annotations', [])]

                if len(category_ids) != len(set(category_ids)):
                    # 중복 발견 시 즉시 삭제
                    os.remove(json_path)
                    
                    deleted_count += 1
                    
                    if deleted_count <= 10:
                        counts = Counter(category_ids)
                        dups = [cid for cid, count in counts.items() if count > 1]
                        print(f"삭제됨: {file} (중복 ID: {dups})")
                        
            except Exception as e:
                print(f"처리 중 오류 발생 ({file}): {e}")


print(f" - 총 검사 파일: {total_checked}개")
print(f" - 삭제된 JSON 파일: {deleted_count}개")

JSON파일과 매치되는 이미지만 남겨놓기

In [ ]:
# 경로 설정
json_merged_dir = r'C:\Users\thsdu\Desktop\8_annotations'
image_root_dir = r'C:\Users\thsdu\Desktop\TS_8_조합.zip'
output_image_dir = r'C:\Users\thsdu\Desktop\8_images'

os.makedirs(output_image_dir, exist_ok=True)

# 필요한 이미지 파일명 목록 추출
target_filenames = [os.path.splitext(f)[0] for f in os.listdir(json_merged_dir) if f.endswith('.json')]
target_set = set(target_filenames)

print(f"추출해야 할 이미지 총 개수: {len(target_set)}개")

# 이미지 찾기 및 복사
copied_count = 0
not_found_count = 0

for target_name in target_filenames:
    folder_prefix = target_name.split('_')[0]
    
    source_folder = os.path.join(image_root_dir, folder_prefix)
    source_file = os.path.join(source_folder, target_name + ".png")
    
    if os.path.exists(source_file):
        target_path = os.path.join(output_image_dir, target_name + ".png")
        shutil.copy2(source_file, target_path)
        copied_count += 1
    else:
        found = False
        for root, _, files in os.walk(image_root_dir):
            if target_name + ".png" in files:
                shutil.copy2(os.path.join(root, target_name + ".png"), os.path.join(output_image_dir, target_name + ".png"))
                copied_count += 1
                found = True
                break
        
        if not found:
            print(f"파일을 찾을 수 없음: {target_name}")
            not_found_count += 1


print(f" - 복사 성공: {copied_count}개")
if not_found_count > 0:
    print(f" - 찾지 못한 이미지: {not_found_count}개")


이미지가 누락되어 JSON파일과 매치되는 이미지가 없을 시 JSON 파일 삭제

In [ ]:
# 경로 설정
json_dir = r'C:\Users\thsdu\Desktop\8_annotations'      # 통합된 JSON 폴더
image_dir = r'C:\Users\thsdu\Desktop\8_images' # 수집된 이미지 폴더

image_files = {os.path.splitext(f)[0] for f in os.listdir(image_dir) if f.endswith('.png')}

print(f"현재 보유 중인 이미지 개수: {len(image_files)}개")

deleted_count = 0
kept_count = 0

for json_file in os.listdir(json_dir):
    if json_file.endswith('.json'):
        json_basename = os.path.splitext(json_file)[0]
        
        if json_basename in image_files:
            kept_count += 1
        else:
            json_path = os.path.join(json_dir, json_file)
            os.remove(json_path)
            deleted_count += 1

print(f" - 유지된 JSON 파일: {kept_count}개")
print(f" - 삭제된 유령 JSON 파일: {deleted_count}개")

bbox 정상적인지 몇개만 확인

In [ ]:
json_dir = r'C:\Users\thsdu\Desktop\8_annotations'  # 통합된 JSON 폴더
img_dir = r'C:\Users\thsdu\Desktop\8_images' # 원본 이미지 폴더

# 시각화할 파일 개수
num_samples = 10
all_jsons = [f for f in os.listdir(json_dir) if f.endswith('.json')]
samples = random.sample(all_jsons, min(num_samples, len(all_jsons)))

for json_file in samples:
    json_path = os.path.join(json_dir, json_file)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 이미지 파일명 추출 및 경로 생성
    img_name = data['images'][0]['file_name']
    img_path = os.path.join(img_dir, img_name)
    
    # 이미지 읽기
    image = cv2.imread(img_path)
    if image is None:
        print(f"이미지를 찾을 수 없음: {img_path}")
        continue
    
    # OpenCV는 BGR을 사용하므로 시각화를 위해 RGB로 변환
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Annotations 순회하며 그리기
    for ann in data.get('annotations', []):
        bbox = ann['bbox'] # [x, y, w, h]
        cat_id = ann['category_id']
        
        x, y, w, h = map(int, bbox)
        
        # 사각형 그리기 (이미지, 좌상단, 우하단, 색상, 두께)
        cv2.rectangle(image, (x, y), (x + w, y + h), (255, 0, 0), 5)
        
        # 클래스 ID 텍스트 쓰기
        cv2.putText(image, str(cat_id), (x, y - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 0, 0), 5)

    # 결과 출력
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.title(f"File: {img_name} (Class IDs: {[ann['category_id'] for ann in data['annotations']]})")
    plt.axis('off')
    plt.show()

Class 당 데이터 갯수 확인

In [ ]:
json_dir = r'C:\Users\thsdu\Desktop\beg_project\all_data\aug_sampling_data\annotations'
output_txt = r"C:\Users\thsdu\Desktop\beg_project\all_data\aug_sampling_data\class_count.txt"

class_counts = {i: 0 for i in range(56)}
total_images = 0
total_objects = 0
processed_count = 0

if not os.path.exists(json_dir):
    print(f"경로가 존재하지 않음: {json_dir}")
else:
    json_files = [f for f in os.listdir(json_dir) if f.lower().endswith('.json')]
    total_to_process = len(json_files)
    print(f"총 {total_to_process}개의 파일")

    # 파일 분석 루프
    for json_file in json_files:
        file_path = os.path.join(json_dir, json_file)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                annotations = data.get('annotations', [])
                for ann in annotations:
                    c_id = ann.get('category_id')
                    if isinstance(c_id, int) and 0 <= c_id <= 55:
                        class_counts[c_id] += 1
                        total_objects += 1
            
            total_images += 1
            processed_count += 1
            
            # 실시간 진행 확인 (500개 단위)
            if processed_count % 500 == 0:
                print(f"[{processed_count}/{total_to_process}]")
        except Exception as e:
            print(f"오류 ({json_file}): {e}")

    # TXT 파일 쓰기
    with open(output_txt, 'w', encoding='utf-8') as f:
        f.write("="*40 + "\n")
        f.write(f"{'Class ID':^10} | {'Count (Bbox)':^15}\n")
        f.write("-" * 40 + "\n")
        
        for i in range(56):
            f.write(f"{i:^10} | {class_counts[i]:^15}\n")
            
        f.write("-" * 40 + "\n")
        f.write(f"총 분석 이미지 수: {total_images} 장\n")
        f.write(f"총 검출된 객체 수: {total_objects} 개\n")
        f.write("="*40 + "\n")



클래스 불균형 시각화

In [ ]:
counts = {
    0: 152, 1: 232, 2: 153, 3: 45, 4: 250, 
    5: 467, 6: 284, 7: 336, 8: 353, 9: 359, 
    10: 452, 11: 344, 12: 276, 13: 822, 14: 592, 
    15: 921, 16: 908, 17: 146, 18: 144, 19: 148, 
    20: 152, 21: 437, 22: 203, 23: 433, 24: 150, 
    25: 596, 26: 462, 27: 172, 28: 206, 29: 159, 
    30: 457, 31: 321, 32: 202, 33: 459, 34: 379, 
    35: 452, 36: 438, 37: 414, 38: 196, 39: 159, 
    40: 440, 41: 190, 42: 190, 43: 577, 44: 437, 
    45: 138, 46: 424, 47: 143, 48: 194, 49: 198, 
    50: 149, 51: 428, 52: 580, 53: 572, 54: 165, 
    55: 149
}

# 분석 결과 요약
ids = list(counts.keys())
values = list(counts.values())

plt.figure(figsize=(15, 5))
plt.bar(ids, values, color='skyblue')
plt.axhline(y=np.mean(values), color='r', linestyle='--', label='Average') # 평균선
plt.xlabel('Class ID')
plt.ylabel('Bbox Count')
plt.title('Pill Class Distribution')
plt.legend()
plt.show()

클래스 불균형 해소를 위한 데이터 증강

In [ ]:
json_dir = r'C:\Users\thsdu\Desktop\beg_project\all_data\train_annotations'
img_dir = r'C:\Users\thsdu\Desktop\beg_project\all_data\train_images'
output_dir = r'C:\Users\thsdu\Desktop\beg_project\all_data\aug_data' # 증강 결과 저장 폴더

# 증강 대상 희귀 클래스 (150개 미만 기준)
rare_classes = [3, 17, 18, 19, 20, 24, 45, 47, 50, 55]

os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'annotations'), exist_ok=True)

def augment_image(img):
    augs = {}
    # 가우시안 노이즈
    noise = np.random.normal(0, 15, img.shape).astype(np.uint8)
    augs['noise'] = cv2.add(img, noise)
    # 블러
    augs['blur'] = cv2.GaussianBlur(img, (5, 5), 0)
    return augs

# 증강 루프 실행
aug_count = 0

for json_file in os.listdir(json_dir): 
    json_path = os.path.join(json_dir, json_file)
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 해당 이미지에 포함된 모든 클래스 ID 추출
    current_ids = [ann['category_id'] for ann in data['annotations']]
    
    # 희귀 클래스가 포함되어 있는지 확인
    if any(cid in rare_classes for cid in current_ids):
        img_name = json_file.replace('.json', '.png')
        img_path = os.path.join(img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            print(f"이미지 로드 실패: {img_name}")
            continue
        
        # 증강 실행
        augmented_results = augment_image(img)
        
        for suffix, aug_img in augmented_results.items():
            base_name = os.path.splitext(img_name)[0]
            new_img_name = f"{base_name}_{suffix}.png"
            new_json_name = f"{base_name}_{suffix}.json"
            
            # 이미지 저장
            cv2.imwrite(os.path.join(output_dir, 'images', new_img_name), aug_img)
            
            # JSON 업데이트
            new_data = data.copy()
            new_data['images'][0]['file_name'] = new_img_name
            with open(os.path.join(output_dir, 'annotations', new_json_name), 'w', encoding='utf-8') as f:
                json.dump(new_data, f, indent=4, ensure_ascii=False)

        aug_count += 1

        


print(f"총 {aug_count * 2}개의 증강 파일이 생성")


증강 이미지 시각화

In [ ]:
aug_root = r'C:\Users\thsdu\Desktop\beg_project\all_data\aug_data'
aug_img_dir = os.path.join(aug_root, 'images')
aug_json_dir = os.path.join(aug_root, 'annotations')

num_samples = 10
sample_files = random.sample(os.listdir(aug_json_dir), min(num_samples, len(os.listdir(aug_json_dir))))

plt.figure(figsize=(20, 10))

for i, json_file in enumerate(sample_files):
    json_path = os.path.join(aug_json_dir, json_file)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    img_name = data['images'][0]['file_name']
    img_path = os.path.join(aug_img_dir, img_name)
    
    img_array = np.fromfile(img_path, np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for ann in data['annotations']:
        x, y, w, h = map(int, ann['bbox'])
        cat_id = ann['category_id']
        
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 8)
        cv2.putText(img, f"ID:{cat_id}", (x, y - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 0, 0), 5)

    plt.subplot(1, num_samples, i + 1)
    plt.imshow(img)
    plt.title(f"Augmented: {img_name.split('_')[-1]}") # suffix(bright, noise 등) 표시
    plt.axis('off')

plt.tight_layout()
plt.show()

이후 클래스별 데이터수 분포

In [ ]:
counts = {
    0: 152, 1: 232, 2: 389, 3: 135, 4: 250, 
    5: 467, 6: 334, 7: 336, 8: 353, 9: 359, 
    10: 452, 11: 344, 12: 330, 13: 1284, 14: 824, 
    15: 921, 16: 908, 17: 438, 18: 432, 19: 444, 
    20: 456, 21: 437, 22: 203, 23: 445, 24: 450, 
    25: 824, 26: 486, 27: 394, 28: 206, 29: 357, 
    30: 475, 31: 321, 32: 202, 33: 483, 34: 379, 
    35: 476, 36: 462, 37: 420, 38: 196, 39: 159, 
    40: 458, 41: 190, 42: 190, 43: 799, 44: 455, 
    45: 414, 46: 430, 47: 429, 48: 194, 49: 198, 
    50: 447, 51: 434, 52: 832, 53: 792, 54: 375, 
    55: 447
}

# 분석 결과 요약
ids = list(counts.keys())
values = list(counts.values())

plt.figure(figsize=(15, 5))
plt.bar(ids, values, color='skyblue')
plt.axhline(y=np.mean(values), color='r', linestyle='--', label='Average') # 평균선
plt.xlabel('Class ID')
plt.ylabel('Bbox Count')
plt.title('Pill Class Distribution')
plt.legend()
plt.show()

In [ ]:
src_img_dir = r'C:\Users\thsdu\Desktop\beg_project\all_data\train_images - 복사본'
src_json_dir = r'C:\Users\thsdu\Desktop\beg_project\all_data\train_annotations - 복사본'
dest_root = r"C:\Users\thsdu\Desktop\beg_project\all_data\aug_sampling_data"


rare_classes = [0, 1, 3, 22, 28, 32, 38, 39, 41, 42, 48, 49] # 200개 이하 소수 클래스
abundant_classes = [13, 14, 15, 16, 25, 43, 52, 53] # 600개 이상 과다 클래스 

os.makedirs(os.path.join(dest_root, 'images'), exist_ok=True)
os.makedirs(os.path.join(dest_root, 'annotations'), exist_ok=True)

stats = {"rare_keep": 0, "middle_keep": 0, "abundant_keep": 0, "deleted": 0}

for json_file in os.listdir(src_json_dir):
    json_path = os.path.join(src_json_dir, json_file)
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    current_ids = [ann['category_id'] for ann in data['annotations']]
    
    if any(cid in rare_classes for cid in current_ids):
        keep_prob = 1.0
        category = "rare_keep"
        
    elif all(cid in abundant_classes for cid in current_ids):
        keep_prob = 0.3
        category = "abundant_keep"
        
    else:
        keep_prob = 0.7
        category = "middle_keep"

    if random.random() < keep_prob:
        img_name = json_file.replace('.json', '.png')
        img_src = os.path.join(src_img_dir, img_name)
        
        if os.path.exists(img_src):
            shutil.copy(img_src, os.path.join(dest_root, 'images', img_name))
            shutil.copy(json_path, os.path.join(dest_root, 'annotations', json_file))
            stats[category] += 1
    else:
        stats["deleted"] += 1
    

print(f"적은 클래스 포함 : {stats['rare_keep']}장")
print(f"적지도 많지도 않은 클래스와 과다클래스로 구성 : {stats['middle_keep']}장")
print(f"많은 클래스로만 구성 : {stats['abundant_keep']}장")
print(f"제외된 이미지 : {stats['deleted']}장")

샘플링까지 마친 후 클래스 분포

In [ ]:
counts = {
    0: 152, 1: 232, 2: 280, 3: 135, 4: 233, 
    5: 315, 6: 238, 7: 301, 8: 248, 9: 331, 
    10: 326, 11: 259, 12: 237, 13: 912, 14: 594, 
    15: 818, 16: 802, 17: 301, 18: 330, 19: 309, 
    20: 317, 21: 316, 22: 203, 23: 307, 24: 315, 
    25: 588, 26: 355, 27: 266, 28: 206, 29: 261, 
    30: 330, 31: 239, 32: 202, 33: 346, 34: 266, 
    35: 331, 36: 314, 37: 288, 38: 196, 39: 159, 
    40: 302, 41: 190, 42: 190, 43: 544, 44: 322, 
    45: 291, 46: 315, 47: 317, 48: 194, 49: 198, 
    50: 320, 51: 308, 52: 609, 53: 544, 54: 288, 
    55: 309
}

# 분석 결과 요약
ids = list(counts.keys())
values = list(counts.values())

plt.figure(figsize=(15, 5))
plt.bar(ids, values, color='skyblue')
plt.axhline(y=np.mean(values), color='r', linestyle='--', label='Average') # 평균선
plt.xlabel('Class ID')
plt.ylabel('Bbox Count')
plt.title('Pill Class Distribution')
plt.legend()
plt.show()

bbox 무결성 검사

In [ ]:
img_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\images'
json_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\annotations'
backup_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\backup'

if not os.path.exists(backup_dir):
    os.makedirs(backup_dir)

json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
invalid_count = 0


for j_file in json_files:
    json_path = os.path.join(json_dir, j_file)
    with open(json_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
        except Exception as e:
            print(f"JSON 파싱 에러 (파일 깨짐): {j_file} -> {e}")
            shutil.move(json_path, os.path.join(backup_dir, j_file))
            invalid_count += 1
            continue

    is_valid = True
    reason = ""

    #  키 존재 여부 및 비어있는지 확인
    if 'annotations' not in data or not data['annotations']:
        is_valid = False
        reason = "객체 데이터(annotations) 없음"
    else:
        for ann in data['annotations']:
            # 2. bbox 키가 아예 없는 경우
            if 'bbox' not in ann:
                is_valid = False
                reason = "bbox 키 누락"
                break
            
            bbox = ann['bbox']
            
            # 3. 좌표 개수가 4개가 아닌 경우 (2개, 3개 등)
            if not isinstance(bbox, list) or len(bbox) != 4:
                is_valid = False
                reason = f"bbox 좌표 개수 오류 (현재 {len(bbox) if isinstance(bbox, list) else 'list 아님'}개)"
                break
            
            # 4. 좌표 값 자체의 유효성 (w, h가 0 이하인 경우)
            x, y, w, h = bbox
            if w <= 0 or h <= 0:
                is_valid = False
                reason = f"유효하지 않은 크기 (w:{w}, h:{h})"
                break
            
    # 문제 파일 격리
    if not is_valid:
        print(f"격리 대상: {j_file} | 사유: {reason}")
        shutil.move(json_path, os.path.join(backup_dir, j_file))
        invalid_count += 1


print(f"격리된 이상 파일: {invalid_count}개")

In [ ]:
img_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\images'
json_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\annotations'

def visualize_local_samples(img_dir, json_dir, num_samples):
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    selected_imgs = random.sample(img_files, num_samples)
    
    # 시각화 설정 (4행 5열)
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))
    axes = axes.flatten()

    for i, img_name in enumerate(selected_imgs):
        img_path = os.path.join(img_dir, img_name)
        json_path = os.path.join(json_dir, os.path.splitext(img_name)[0] + ".json")
        
        # 이미지 열기
        img = Image.open(img_path)
        axes[i].imshow(img)
        
        # JSON 열기 및 Bbox 그리기
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            for ann in data.get('annotations', []):
                # [x, y, w, h] 형식 가정
                bbox = ann['bbox']
                if len(bbox) == 4:
                    x, y, w, h = bbox
                    # 빨간색 박스 추가
                    rect = patches.Rectangle(
                        (x, y), w, h, linewidth=1.5, edgecolor='red', facecolor='none'
                    )
                    axes[i].add_patch(rect)
                    # 클래스 ID 표시
                    axes[i].text(x, y, f"ID:{ann['category_id']}", color='yellow', 
                                 fontsize=8, backgroundcolor='black')
        
        axes[i].set_title(img_name, fontsize=8)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# 실행
visualize_local_samples(img_dir, json_dir, num_samples=20)

In [ ]:
img_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\images'
json_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\annotations'
corrupted_dir = r'C:\Users\thsdu\Desktop\beg_project\Datas\aug_sampling_data\backup'


def check_image_integrity():
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    bad_count = 0
    
    
    for img_name in img_files:
        img_path = os.path.join(img_dir, img_name)
        try:
            with Image.open(img_path) as img:
                img.verify() # 1차 검사: 파일 구조 확인
            
            # verify만으로 부족할 때가 있어서 실제로 load까지 해봅니다.
            with Image.open(img_path) as img:
                img.load() 
                
        except (IOError, SyntaxError, OSError) as e:
            print(f"깨진 이미지 발견: {img_name} | 사유: {e}")
            # 깨진 이미지와 대응하는 JSON을 격리 폴더로 이동
            shutil.move(img_path, os.path.join(corrupted_dir, img_name))
            
            json_name = os.path.splitext(img_name)[0] + ".json"
            json_path = os.path.join(json_dir, json_name)
            if os.path.exists(json_path):
                shutil.move(json_path, os.path.join(corrupted_dir, json_name))
            
            bad_count += 1

    print(f"깨진 파일 {bad_count}개를 격리")

check_image_integrity()

이미지와 bbox 간의 문제가 있는 것들 확인 후 격리
여기서 지금 막히고 있어서 진행중인코드는 잘랐습니다.